# XP Ninja — Build an Intelligent Document Assistant

## Solution académique complète

Ce notebook construit un assistant documentaire spécialisé dans l’analyse
d’un extrait de contrat de services.

L’assistant peut :

- extraire les clauses importantes ;
- produire un résumé en langage simple ;
- répondre à des questions de suivi ;
- conserver une mémoire structurée ;
- vérifier et corriger ses propres réponses ;
- fonctionner en mode LLM avec Ollama ou en mode local déterministe.

Le code Python est commenté pour rendre le workflow compréhensible.

## Objectifs pédagogiques

À la fin du notebook, vous saurez :

1. combiner role prompting, contraintes et exemples ;
2. concevoir un prompt de résumé fondé sur des preuves ;
3. adapter une réponse au type de question juridique ;
4. transmettre le contexte entre plusieurs tours ;
5. construire un pipeline dynamique avec logique conditionnelle ;
6. limiter les hallucinations par grounding et critique ;
7. distinguer information contractuelle et conseil juridique ;
8. implémenter une mémoire conversationnelle minimale.

## Architecture du workflow

```text
Contrat
   ↓
Extraction des clauses et faits
   ↓
Résumé structuré
   ↓
Mémoire documentaire
   ↓
Question utilisateur
   ↓
Prompt adaptatif + passages pertinents
   ↓
Réponse initiale
   ↓
Critique de fidélité et de clarté
   ↓
Réponse révisée + mise à jour de la mémoire
```

La réponse finale doit toujours rester fondée sur le texte fourni.

## Note sur le raisonnement

Les prompts du notebook utilisent un raisonnement **court et vérifiable** :

- identifier la clause ;
- extraire les faits ;
- effectuer un calcul explicite si nécessaire ;
- signaler les hypothèses ;
- produire la réponse.

Le système ne demande pas au modèle de révéler un raisonnement interne
exhaustif. Il demande plutôt une justification concise et des références au
texte.

# 0. Imports et document source

In [ ]:
import json
import os
import re
import subprocess
from dataclasses import asdict, dataclass, field
from datetime import datetime
from typing import Any, Dict, List, Optional, Tuple

In [ ]:
# Keep the contract in one canonical variable.
# All prompts and deterministic functions use this exact source.
CONTRACT_TEXT = """
Service Agreement – Excerpt

This Service Agreement (“Agreement”) is made effective as of March 1, 2025,
by and between BrightLine Technologies Ltd., hereinafter referred to as
“Provider”, and NovaWare Systems Inc., hereinafter referred to as “Client”.

Scope of Work: Provider shall deliver cloud infrastructure management
services, including monitoring, incident response, and monthly reporting,
as described in Exhibit A.

Payment Terms: Client agrees to pay a fixed monthly fee of $12,000, payable
within 30 days of receipt of invoice. Late payments will incur a 2% penalty
per month.

Term and Termination: This Agreement shall commence on March 1, 2025, and
remain in effect for 12 months. Either party may terminate with 30 days’
written notice.

Confidentiality: Both parties agree to protect the confidentiality of
proprietary or sensitive information shared during the course of the
engagement.

Limitation of Liability: Provider’s total liability shall not exceed the
fees paid by Client in the 3 months prior to a claim. Provider is not liable
for indirect or consequential damages.

Governing Law: This Agreement shall be governed by the laws of the State of
California.
""".strip()

print(CONTRACT_TEXT)

# Helper — Exécuter un prompt avec Ollama

Cette fonction est optionnelle.

- Si Ollama est installé, elle envoie le prompt au modèle local.
- Dans Google Colab, Ollama est généralement absent.
- Dans ce cas, le notebook affiche le prompt en mode `dry-run` sans échouer.

In [ ]:
def run_prompt(
    prompt: str,
    model: Optional[str] = None,
) -> Optional[str]:
    """Run a single prompt through Ollama when available.

    Parameters
    ----------
    prompt:
        Complete prompt sent to the language model.
    model:
        Ollama model name. When omitted, the OLLAMA_MODEL environment
        variable is used, then 'llama3' as fallback.

    Returns
    -------
    Optional[str]
        Model output, or None when Ollama is unavailable.
    """
    if not prompt or not prompt.strip():
        raise ValueError("The prompt cannot be empty.")

    selected_model = model or os.environ.get(
        "OLLAMA_MODEL",
        "llama3",
    )

    try:
        # Send the prompt through standard input and capture the response.
        process = subprocess.run(
            ["ollama", "run", selected_model],
            input=prompt.encode("utf-8"),
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            check=True,
        )

        output = process.stdout.decode(
            "utf-8",
            errors="ignore",
        ).strip()

        print(output)
        return output

    except FileNotFoundError:
        print("[dry-run] Ollama is not installed.")
        print("\nPrompt that would be sent:\n")
        print(prompt)

    except subprocess.CalledProcessError as error:
        error_text = (
            error.stderr.decode("utf-8", errors="ignore")
            if error.stderr
            else str(error)
        )
        print("[dry-run] Ollama call failed:", error_text)
        print("\nPrompt that would be sent:\n")
        print(prompt)

    return None

# 1. Prétraitement et extraction déterministe

Même avec un LLM, il est utile d’extraire les sections connues avec du code.

Cela permet :

- de vérifier les sorties du modèle ;
- de transmettre uniquement les clauses pertinentes ;
- de réduire le bruit ;
- de calculer les montants sans hallucination.

In [ ]:
# Define the headings expected in the contract.
CLAUSE_HEADINGS = [
    "Scope of Work",
    "Payment Terms",
    "Term and Termination",
    "Confidentiality",
    "Limitation of Liability",
    "Governing Law",
]


def normalize_whitespace(text: str) -> str:
    """Collapse repeated whitespace while preserving the wording."""
    return re.sub(r"\s+", " ", str(text or "")).strip()


def extract_contract_sections(
    contract_text: str,
) -> Dict[str, str]:
    """Split the contract into its introductory text and named clauses.

    The function searches for the known headings and takes the text between
    consecutive headings. This is deterministic and easy to audit.
    """
    cleaned_text = normalize_whitespace(contract_text)

    # Build one regular expression containing every known heading.
    heading_pattern = "|".join(
        re.escape(heading)
        for heading in CLAUSE_HEADINGS
    )

    matches = list(
        re.finditer(
            rf"({heading_pattern}):\s*",
            cleaned_text,
        )
    )

    sections: Dict[str, str] = {}

    if not matches:
        return {"Full Document": cleaned_text}

    # Save the introductory party and effective-date paragraph.
    sections["Introduction"] = cleaned_text[:matches[0].start()].strip()

    # Extract the text located after each heading.
    for index, match in enumerate(matches):
        heading = match.group(1)
        start = match.end()
        end = (
            matches[index + 1].start()
            if index + 1 < len(matches)
            else len(cleaned_text)
        )
        sections[heading] = cleaned_text[start:end].strip()

    return sections


contract_sections = extract_contract_sections(CONTRACT_TEXT)

for heading, content in contract_sections.items():
    print(f"\n[{heading}]")
    print(content)

## 1.1 Extraction des faits structurés

In [ ]:
def first_regex_group(
    pattern: str,
    text: str,
    flags: int = re.IGNORECASE,
) -> Optional[str]:
    """Return the first captured regex group or None."""
    match = re.search(pattern, text, flags)
    return match.group(1) if match else None


def parse_currency_to_cents(value: str) -> int:
    """Convert a currency string such as '$12,000' to integer cents."""
    numeric_value = re.sub(r"[^0-9.]", "", value)
    return int(round(float(numeric_value) * 100))


def extract_contract_facts(
    contract_text: str,
) -> Dict[str, Any]:
    """Extract reusable factual fields from the contract excerpt."""
    text = normalize_whitespace(contract_text)

    monthly_fee_text = first_regex_group(
        r"monthly fee of (\$\d{1,3}(?:,\d{3})*(?:\.\d{2})?)",
        text,
    )

    facts = {
        "provider": first_regex_group(
            r"between ([A-Za-z0-9 .,&-]+?), hereinafter referred to as [“\"]Provider",
            text,
        ),
        "client": first_regex_group(
            r"and ([A-Za-z0-9 .,&-]+?), hereinafter referred to as [“\"]Client",
            text,
        ),
        "effective_date": first_regex_group(
            r"effective as of ([A-Z][a-z]+ \d{1,2}, \d{4})",
            text,
        ),
        "monthly_fee_text": monthly_fee_text,
        "monthly_fee_cents": (
            parse_currency_to_cents(monthly_fee_text)
            if monthly_fee_text
            else None
        ),
        "invoice_due_days": first_regex_group(
            r"within (\d+) days of receipt of invoice",
            text,
        ),
        "late_penalty_percent_per_month": first_regex_group(
            r"(\d+(?:\.\d+)?)% penalty per month",
            text,
        ),
        "term_months": first_regex_group(
            r"remain in effect for (\d+) months",
            text,
        ),
        "termination_notice_days": first_regex_group(
            r"terminate with (\d+) days[’'] written notice",
            text,
        ),
        "liability_lookback_months": first_regex_group(
            r"fees paid by Client in the (\d+) months prior to a claim",
            text,
        ),
        "governing_law": first_regex_group(
            r"laws of the (State of [A-Za-z ]+)",
            text,
        ),
    }

    # Convert numeric strings into integers or floats when available.
    integer_fields = [
        "invoice_due_days",
        "term_months",
        "termination_notice_days",
        "liability_lookback_months",
    ]

    for field_name in integer_fields:
        if facts[field_name] is not None:
            facts[field_name] = int(facts[field_name])

    if facts["late_penalty_percent_per_month"] is not None:
        facts["late_penalty_percent_per_month"] = float(
            facts["late_penalty_percent_per_month"]
        )

    return facts


contract_facts = extract_contract_facts(CONTRACT_TEXT)

print(json.dumps(contract_facts, indent=2))

# Step 1 — Initial Summary Prompt

Le prompt utilise un pattern **few-shot + extraction vérifiable**.

Il demande au modèle :

1. d’identifier les clauses nécessaires ;
2. de résumer en langage simple ;
3. de ne pas ajouter de droits ou obligations absents ;
4. de citer les sections utilisées.

In [ ]:
summary_prompt = f"""
Act as a careful contract summarization assistant.

Your job is to explain a contract excerpt in plain English for a business
reader. This is educational contract analysis, not legal advice.

Follow this method:
1. Identify only the clauses relevant to responsibilities, payment,
   termination, confidentiality, liability, and governing law.
2. Extract the exact duties, amounts, dates, notice periods, percentages,
   and liability limits.
3. Translate them into plain English without changing their legal meaning.
4. Verify every statement against the document.
5. If a requested point is not stated, write "Not stated in the excerpt."

Mini-example:

Source clause:
"Customer shall pay $500 within 15 days. Either party may terminate with
10 days' written notice."

Good plain-English summary:
- Payment: The customer must pay $500 within 15 days.
- Termination: Either side can end the agreement by giving 10 days'
  written notice.

Do not:
- invent remedies, exceptions, renewal rights, or legal interpretations;
- treat Exhibit A as available when it is not included;
- calculate a liability amount without clearly stating the assumption.

Contract:
<contract>
{CONTRACT_TEXT}
</contract>

Required output:

1. Parties and purpose
2. Provider responsibilities
3. Client responsibilities
4. Payment terms
5. Term and termination
6. Confidentiality
7. Liability limitations
8. Governing law
9. Missing or referenced material
10. Plain-English takeaway — maximum 3 sentences

For each numbered section:
- give a concise explanation;
- name the supporting clause in parentheses.

Keep the complete answer under 450 words.
""".strip()

print(summary_prompt)

## 1.1 Résumé déterministe de référence

Ce résumé sert de sortie attendue et de mémoire initiale, même lorsqu’aucun
LLM n’est disponible.

In [ ]:
def format_currency_from_cents(cents: Optional[int]) -> str:
    """Format integer cents as US dollars."""
    if cents is None:
        return "Not stated"
    return f"${cents / 100:,.2f}"


def build_reference_summary(
    sections: Dict[str, str],
    facts: Dict[str, Any],
) -> str:
    """Create a grounded plain-English summary from extracted facts."""
    monthly_fee = format_currency_from_cents(
        facts.get("monthly_fee_cents")
    )

    return (
        f"1. Parties and purpose: {facts.get('provider')} is the Provider "
        f"and {facts.get('client')} is the Client. The agreement covers "
        "cloud infrastructure management services. "
        "(Introduction; Scope of Work)\n\n"
        "2. Provider responsibilities: The Provider must perform monitoring, "
        "incident response, and monthly reporting. Further details are "
        "referenced in Exhibit A, which is not included here. "
        "(Scope of Work)\n\n"
        f"3. Client responsibilities: The Client must pay {monthly_fee} "
        f"per month within {facts.get('invoice_due_days')} days after "
        "receiving an invoice. Both parties must protect confidential "
        "information. (Payment Terms; Confidentiality)\n\n"
        f"4. Payment terms: Late payments incur a "
        f"{facts.get('late_penalty_percent_per_month'):g}% penalty per "
        "month. (Payment Terms)\n\n"
        f"5. Term and termination: The agreement starts on "
        f"{facts.get('effective_date')} and lasts "
        f"{facts.get('term_months')} months. Either party may terminate "
        f"with {facts.get('termination_notice_days')} days' written notice. "
        "(Term and Termination)\n\n"
        "6. Confidentiality: Each party must protect proprietary or "
        "sensitive information shared during the engagement. "
        "(Confidentiality)\n\n"
        f"7. Liability limitations: The Provider's total liability is "
        f"capped at the fees actually paid by the Client during the "
        f"{facts.get('liability_lookback_months')} months before a claim. "
        "The Provider excludes liability for indirect or consequential "
        "damages. (Limitation of Liability)\n\n"
        f"8. Governing law: {facts.get('governing_law')} law governs the "
        "agreement. (Governing Law)\n\n"
        "9. Missing or referenced material: Exhibit A is referenced but "
        "not included, so the complete service specifications are not "
        "available.\n\n"
        "10. Plain-English takeaway: BrightLine manages NovaWare's cloud "
        "infrastructure for a fixed monthly fee. Either side can terminate "
        "with notice, and the Provider has a limited damages exposure."
    )


reference_summary = build_reference_summary(
    contract_sections,
    contract_facts,
)

print(reference_summary)

## 1.2 Optional LLM test

In [ ]:
# This cell runs only when Ollama is available.
# Otherwise, the prompt is printed in dry-run mode.
summary_llm_output = run_prompt(summary_prompt)

# Step 2 — Role-Based Follow-Up Q&A

Le rôle demandé est celui d’un contract lawyer.

Pour réduire les risques :

- le rôle est utilisé pour l’analyse pédagogique ;
- le modèle ne doit pas présenter sa réponse comme un avis juridique ;
- il doit séparer le texte, les calculs illustratifs et les interprétations ;
- il doit signaler les informations manquantes.

In [ ]:
qa_prompt_template = """
Act as a contract lawyer providing an educational explanation of the
supplied agreement excerpt. Do not create an attorney-client relationship
and do not present the response as jurisdiction-specific legal advice.

Document:
<contract>
{contract}
</contract>

Verified summary:
<summary>
{summary}
</summary>

User question:
<question>
{question}
</question>

Use an instance-adaptive response method:

A. First classify the question internally as one of:
   - clause explanation;
   - numerical calculation;
   - rights and obligations;
   - risk or ambiguity;
   - information not present.

B. Apply the matching method:
   - Clause explanation: quote or closely paraphrase the relevant clause,
     then explain it in plain English.
   - Numerical calculation: identify the formula, show the short
     calculation, and state all assumptions.
   - Rights and obligations: identify which party has the duty or right.
   - Risk or ambiguity: distinguish the literal text from possible legal
     interpretation and recommend professional review.
   - Missing information: state that the excerpt does not answer the
     question.

C. Verify:
   - Every factual claim must be supported by the excerpt.
   - Do not invent California law, case law, remedies, or exceptions.
   - Do not assume Exhibit A's contents.
   - Preserve words such as "shall", "may", "not exceed", and "written".
   - If the answer depends on facts outside the excerpt, say so.

Output format:
1. Direct answer — 1 to 3 sentences.
2. What the clause says — concise evidence from the document.
3. Practical meaning — plain English.
4. Assumptions or missing information — when applicable.
5. Caution — one sentence stating that the explanation is not legal advice.

Keep the answer under 260 words.
""".strip()


def build_qa_prompt(
    question: str,
    summary: str = reference_summary,
) -> str:
    """Insert the document, memory summary, and user question."""
    if not question or not question.strip():
        raise ValueError("The question cannot be empty.")

    return qa_prompt_template.format(
        contract=CONTRACT_TEXT,
        summary=summary,
        question=question.strip(),
    )


example_question = (
    "Can you explain the limitation of liability clause?"
)

qa_prompt = build_qa_prompt(example_question)
print(qa_prompt)

## 2.1 Q&A déterministe

La fonction suivante répond aux thèmes principaux sans modèle externe.

Elle constitue :

- un fallback fonctionnel ;
- un oracle simple pour vérifier une sortie LLM ;
- un exemple de logique adaptative.

In [ ]:
def illustrative_liability_cap(
    facts: Dict[str, Any],
) -> Optional[int]:
    """Calculate an illustrative cap using the stated monthly fee.

    Important:
    The contract caps liability at fees actually paid, not automatically
    at monthly fee × three. The multiplication is only an illustration
    when three full monthly fees were paid.
    """
    monthly_fee = facts.get("monthly_fee_cents")
    months = facts.get("liability_lookback_months")

    if monthly_fee is None or months is None:
        return None

    return monthly_fee * months


def answer_contract_question(
    question: str,
    sections: Dict[str, str],
    facts: Dict[str, Any],
) -> str:
    """Answer common questions using only extracted contract content."""
    normalized_question = question.lower()

    if "liability" in normalized_question:
        illustrative_cap = illustrative_liability_cap(facts)
        cap_text = format_currency_from_cents(illustrative_cap)

        return (
            "Direct answer: The Provider's total liability is limited to "
            f"the fees the Client actually paid during the "
            f"{facts['liability_lookback_months']} months before the claim. "
            "The Provider also excludes indirect and consequential damages.\n\n"
            "What the clause says: Liability may not exceed the prior "
            "three months of fees paid.\n\n"
            f"Practical meaning: If three full monthly fees of "
            f"{facts['monthly_fee_text']} were actually paid, the "
            f"illustrative cap would be {cap_text}. The real cap still "
            "depends on the amount actually paid during that period.\n\n"
            "Caution: This is a plain-English explanation, not legal advice."
        )

    if "terminate" in normalized_question or "termination" in normalized_question:
        return (
            f"Direct answer: Either party may terminate the agreement by "
            f"giving {facts['termination_notice_days']} days' written "
            "notice.\n\n"
            "What the clause says: The excerpt provides a general notice "
            "right for both parties. It does not state an immediate "
            "termination right for late payment or breach.\n\n"
            "Caution: The complete agreement or applicable law may contain "
            "additional rules not shown here."
        )

    if "late" in normalized_question or "penalty" in normalized_question:
        return (
            f"Direct answer: A late payment incurs a "
            f"{facts['late_penalty_percent_per_month']:g}% penalty per "
            "month.\n\n"
            f"The monthly fee is {facts['monthly_fee_text']} and is due "
            f"within {facts['invoice_due_days']} days after the invoice is "
            "received.\n\n"
            "The excerpt does not state that late payment automatically "
            "causes immediate termination."
        )

    if "payment" in normalized_question or "fee" in normalized_question:
        return (
            f"Direct answer: The Client must pay "
            f"{facts['monthly_fee_text']} per month within "
            f"{facts['invoice_due_days']} days after receiving an invoice. "
            f"Late payments carry a "
            f"{facts['late_penalty_percent_per_month']:g}% monthly penalty."
        )

    if "responsib" in normalized_question or "scope" in normalized_question:
        return (
            "Direct answer: The Provider must manage cloud infrastructure, "
            "including monitoring, incident response, and monthly reporting. "
            "The full service details are said to be in Exhibit A, which is "
            "not included in the excerpt."
        )

    if "confidential" in normalized_question:
        return (
            "Direct answer: Both parties must protect proprietary or "
            "sensitive information shared during the engagement. The excerpt "
            "does not define detailed security measures, exceptions, or a "
            "survival period."
        )

    if "law" in normalized_question or "california" in normalized_question:
        return (
            f"Direct answer: The agreement is governed by "
            f"{facts['governing_law']} law. The excerpt does not specify a "
            "court, venue, or dispute-resolution procedure."
        )

    return (
        "The requested information is not clearly stated in the supplied "
        "excerpt. A reliable answer would require the complete agreement or "
        "additional facts. This is not legal advice."
    )


deterministic_answer = answer_contract_question(
    example_question,
    contract_sections,
    contract_facts,
)

print(deterministic_answer)

## 2.2 Vérification du calcul illustratif

In [ ]:
# Use cents to avoid floating-point money errors.
expected_cap_cents = 12_000 * 100 * 3
calculated_cap_cents = illustrative_liability_cap(contract_facts)

assert calculated_cap_cents == expected_cap_cents

print(
    "Illustrative three-month cap:",
    format_currency_from_cents(calculated_cap_cents),
)
print(
    "Reminder: the contractual wording uses fees actually paid, "
    "so the real cap may differ."
)

# Step 3 — Memory Integration

## Technique choisie : Structured Conversation History

La mémoire conserve séparément :

- le résumé vérifié ;
- les faits extraits ;
- les questions et réponses récentes ;
- les points non résolus ;
- les préférences de présentation.

Cette méthode est plus contrôlable que le simple passage de toute la
conversation brute.

In [ ]:
@dataclass
class ConversationTurn:
    """One user-assistant exchange stored in memory."""

    user_question: str
    assistant_answer: str
    relevant_clauses: List[str]
    timestamp: str


@dataclass
class DocumentAssistantMemory:
    """Structured memory for the document assistant."""

    document_name: str
    verified_summary: str
    key_facts: Dict[str, Any]
    recent_turns: List[ConversationTurn] = field(default_factory=list)
    unresolved_questions: List[str] = field(default_factory=list)
    user_preferences: Dict[str, Any] = field(
        default_factory=lambda: {
            "language": "English",
            "detail_level": "plain English",
            "include_clause_names": True,
        }
    )


assistant_memory = DocumentAssistantMemory(
    document_name="Service Agreement – Excerpt",
    verified_summary=reference_summary,
    key_facts=contract_facts,
)

print(
    json.dumps(
        asdict(assistant_memory),
        indent=2,
        default=str,
    )
)

## 3.1 Ajouter un tour à la mémoire

In [ ]:
def infer_relevant_clauses(question: str) -> List[str]:
    """Map question keywords to likely contract clauses."""
    normalized = question.lower()
    clauses: List[str] = []

    keyword_map = {
        "Payment Terms": [
            "payment",
            "fee",
            "invoice",
            "late",
            "penalty",
        ],
        "Term and Termination": [
            "terminate",
            "termination",
            "notice",
            "term",
        ],
        "Limitation of Liability": [
            "liability",
            "damages",
            "cap",
        ],
        "Scope of Work": [
            "scope",
            "service",
            "monitoring",
            "incident",
            "reporting",
        ],
        "Confidentiality": [
            "confidential",
            "proprietary",
            "sensitive",
        ],
        "Governing Law": [
            "law",
            "california",
            "jurisdiction",
        ],
    }

    for clause_name, keywords in keyword_map.items():
        if any(keyword in normalized for keyword in keywords):
            clauses.append(clause_name)

    return clauses or ["Introduction"]


def add_memory_turn(
    memory: DocumentAssistantMemory,
    question: str,
    answer: str,
    max_turns: int = 5,
) -> None:
    """Store one turn and keep only the most recent exchanges."""
    turn = ConversationTurn(
        user_question=question,
        assistant_answer=answer,
        relevant_clauses=infer_relevant_clauses(question),
        timestamp=datetime.utcnow().isoformat(timespec="seconds") + "Z",
    )

    memory.recent_turns.append(turn)

    # Limit context size so memory does not grow forever.
    memory.recent_turns = memory.recent_turns[-max_turns:]


add_memory_turn(
    assistant_memory,
    example_question,
    deterministic_answer,
)

print(
    json.dumps(
        asdict(assistant_memory),
        indent=2,
        default=str,
    )
)

## 3.2 Construire le prompt contextuel

In [ ]:
memory_prompt_template = """
Act as a grounded contract-document assistant.

Canonical document:
<contract>
{contract}
</contract>

Verified document memory:
<memory>
{memory_json}
</memory>

New user question:
<question>
{question}
</question>

Instructions:
1. Use the original contract as the highest-priority source.
2. Use memory only to preserve continuity, not to override the document.
3. Refer to earlier answers when useful, but correct them if they conflict
   with the contract.
4. Answer the new question directly.
5. State whether the answer is explicit, calculated, inferred, or not
   available.
6. For calculations, show a short formula and the assumptions.
7. Do not invent legal rules, Exhibit A content, or facts outside the text.
8. End with a one-sentence non-legal-advice caution.

Return:
- Direct answer
- Relevant clause
- Connection to prior context
- Assumptions or missing information
""".strip()


def build_memory_prompt(
    memory: DocumentAssistantMemory,
    question: str,
) -> str:
    """Serialize structured memory into the next-turn prompt."""
    memory_json = json.dumps(
        asdict(memory),
        indent=2,
        default=str,
    )

    return memory_prompt_template.format(
        contract=CONTRACT_TEXT,
        memory_json=memory_json,
        question=question.strip(),
    )


follow_up_question = (
    "Does that mean the Provider can terminate immediately if we pay late?"
)

memory_prompt = build_memory_prompt(
    assistant_memory,
    follow_up_question,
)

print(memory_prompt)

## 3.3 Réponse contextuelle de référence

In [ ]:
follow_up_answer = answer_contract_question(
    follow_up_question,
    contract_sections,
    contract_facts,
)

add_memory_turn(
    assistant_memory,
    follow_up_question,
    follow_up_answer,
)

print(follow_up_answer)

## 3.4 Pourquoi cette réponse est cohérente

La mémoire contient déjà une explication de la responsabilité, mais la
nouvelle question porte sur :

- les paiements tardifs ;
- la résiliation.

Le système récupère donc les clauses **Payment Terms** et
**Term and Termination**.

Il ne fusionne pas automatiquement la pénalité de retard avec un droit de
résiliation immédiate, car cette conséquence n’est pas écrite.

# Step 4 — Mitigation and Refinement

La critique vérifie :

1. la fidélité au document ;
2. les chiffres et calculs ;
3. la séparation entre texte et interprétation ;
4. les informations manquantes ;
5. la clarté ;
6. le caractère non juridique du conseil.

In [ ]:
critique_prompt_template = """
Act as an independent contract-answer reviewer.

Original contract:
<contract>
{contract}
</contract>

User question:
<question>
{question}
</question>

Draft answer:
<draft_answer>
{draft_answer}
</draft_answer>

Review the draft using this checklist:

A. Grounding
- Is every factual statement supported by the contract?
- Does the draft invent a clause, remedy, exception, deadline, or Exhibit A
  detail?

B. Numerical accuracy
- Are all amounts, percentages, months, and notice periods correct?
- Are derived numbers clearly labelled as illustrations?
- Does the answer distinguish fees stated from fees actually paid?

C. Legal precision
- Does the answer preserve "may", "shall", "written notice", and
  "not exceed"?
- Does it avoid presenting general California law as part of the excerpt?
- Does it avoid creating an attorney-client relationship?

D. Completeness and clarity
- Does it answer the user's actual question?
- Does it identify missing information?
- Is it understandable to a non-lawyer?

Return valid JSON only:
{{
  "verdict": "pass" or "revise",
  "grounding_issues": [],
  "numerical_issues": [],
  "clarity_issues": [],
  "missing_caveats": [],
  "corrected_answer": "full revised answer or null"
}}
""".strip()


def build_critique_prompt(
    question: str,
    draft_answer: str,
) -> str:
    """Build a second-pass prompt for independent review."""
    return critique_prompt_template.format(
        contract=CONTRACT_TEXT,
        question=question,
        draft_answer=draft_answer,
    )


critique_prompt = build_critique_prompt(
    example_question,
    deterministic_answer,
)

print(critique_prompt)

## 4.1 Audit déterministe d’une réponse

In [ ]:
def audit_contract_answer(
    question: str,
    answer: str,
    facts: Dict[str, Any],
) -> Dict[str, Any]:
    """Run basic deterministic checks on a contract answer.

    This audit does not replace legal review or semantic fact-checking.
    It catches several high-value errors from this specific exercise.
    """
    normalized_answer = answer.lower()

    issues: List[str] = []
    warnings: List[str] = []

    # Detect a common unsupported conclusion in this contract.
    if "immediate termination" in normalized_answer:
        if "does not" not in normalized_answer and "not state" not in normalized_answer:
            issues.append(
                "The excerpt does not grant an explicit immediate "
                "termination right."
            )

    # Verify key numerical values when their topics are discussed.
    if "late" in question.lower() or "penalty" in question.lower():
        expected_penalty = (
            f"{facts['late_penalty_percent_per_month']:g}%"
        )
        if expected_penalty not in answer:
            warnings.append(
                f"The answer may omit the stated {expected_penalty} penalty."
            )

    if "liability" in question.lower():
        expected_months = str(
            facts["liability_lookback_months"]
        )
        if expected_months not in answer:
            warnings.append(
                "The answer may omit the three-month lookback."
            )

        # A $36,000 calculation must be described as illustrative.
        if "$36,000" in answer and "illustrative" not in normalized_answer:
            issues.append(
                "$36,000 must be labelled as an illustration based on "
                "three full monthly fees actually being paid."
            )

    if "legal advice" not in normalized_answer:
        warnings.append(
            "Add a non-legal-advice caution for a legal-document assistant."
        )

    return {
        "verdict": "revise" if issues else "pass_with_warnings" if warnings else "pass",
        "issues": issues,
        "warnings": warnings,
    }


deterministic_audit = audit_contract_answer(
    example_question,
    deterministic_answer,
    contract_facts,
)

print(json.dumps(deterministic_audit, indent=2))

## 4.2 Multi-agent critique optionnel

Le workflow peut séparer trois rôles :

1. **Answerer** : produit une réponse fondée sur les clauses ;
2. **Reviewer** : cherche les erreurs et omissions ;
3. **Editor** : révise uniquement les éléments signalés.

Séparer les rôles réduit le risque qu’un même modèle confirme sans recul sa
première réponse.

In [ ]:
revision_prompt_template = """
Act as the final contract-answer editor.

Contract:
<contract>
{contract}
</contract>

User question:
<question>
{question}
</question>

Draft answer:
<draft>
{draft}
</draft>

Reviewer findings:
<review>
{review}
</review>

Revise the answer only as needed to fix supported reviewer findings.

Requirements:
- Use only the contract excerpt.
- Preserve accurate parts of the draft.
- Correct unsupported claims and calculations.
- Label assumptions and illustrative calculations.
- State when information is unavailable.
- Use plain English.
- End with a brief non-legal-advice caution.
- Return only the revised answer.
""".strip()


def build_revision_prompt(
    question: str,
    draft_answer: str,
    review: Dict[str, Any],
) -> str:
    """Create the final editor prompt from answer and critique."""
    return revision_prompt_template.format(
        contract=CONTRACT_TEXT,
        question=question,
        draft=draft_answer,
        review=json.dumps(review, indent=2),
    )


revision_prompt = build_revision_prompt(
    example_question,
    deterministic_answer,
    deterministic_audit,
)

print(revision_prompt)

# 5. Complete Intelligent Document Assistant

La classe suivante réunit :

- le document ;
- l’extraction ;
- le résumé ;
- les réponses ;
- la mémoire ;
- l’audit.

Elle fonctionne sans LLM et peut servir de base à une intégration Ollama,
Gemini, OpenAI ou Hugging Face.

In [ ]:
class IntelligentDocumentAssistant:
    """Grounded assistant for one contract document."""

    def __init__(self, document_name: str, document_text: str):
        # Store the canonical source.
        self.document_name = document_name
        self.document_text = document_text

        # Build deterministic representations used for grounding.
        self.sections = extract_contract_sections(document_text)
        self.facts = extract_contract_facts(document_text)
        self.summary = build_reference_summary(
            self.sections,
            self.facts,
        )

        # Initialize structured memory.
        self.memory = DocumentAssistantMemory(
            document_name=document_name,
            verified_summary=self.summary,
            key_facts=self.facts,
        )

    def get_summary(self) -> str:
        """Return the verified plain-English summary."""
        return self.summary

    def ask(self, question: str) -> Dict[str, Any]:
        """Answer, audit, and store a user question."""
        if not question or not question.strip():
            raise ValueError("The question cannot be empty.")

        answer = answer_contract_question(
            question,
            self.sections,
            self.facts,
        )

        audit = audit_contract_answer(
            question,
            answer,
            self.facts,
        )

        add_memory_turn(
            self.memory,
            question,
            answer,
        )

        return {
            "question": question,
            "answer": answer,
            "audit": audit,
            "relevant_clauses": infer_relevant_clauses(question),
        }

    def build_llm_prompt(self, question: str) -> str:
        """Create a context-aware prompt for an external LLM."""
        return build_memory_prompt(
            self.memory,
            question,
        )

    def export_memory(self) -> Dict[str, Any]:
        """Return JSON-serializable memory for storage or inspection."""
        return asdict(self.memory)

## 5.1 Démonstration complète

In [ ]:
assistant = IntelligentDocumentAssistant(
    document_name="Service Agreement – Excerpt",
    document_text=CONTRACT_TEXT,
)

print("SUMMARY")
print("=" * 80)
print(assistant.get_summary())

demo_questions = [
    "When is the monthly fee due?",
    "Can either party terminate the agreement?",
    "What is the maximum liability?",
    "What does the contract say about data breach notification?",
]

demo_results = []

for question in demo_questions:
    result = assistant.ask(question)
    demo_results.append(result)

    print("\n" + "=" * 80)
    print("QUESTION:", question)
    print("\nANSWER:")
    print(result["answer"])
    print("\nAUDIT:")
    print(json.dumps(result["audit"], indent=2))

## 5.2 Mémoire après plusieurs tours

In [ ]:
exported_memory = assistant.export_memory()

print(
    json.dumps(
        exported_memory,
        indent=2,
        default=str,
    )
)

# The assistant stores only the last five turns by design.
assert len(exported_memory["recent_turns"]) <= 5

# 6. Tests automatiques

Ces tests valident les faits centraux du contrat et plusieurs règles
anti-hallucination.

In [ ]:
# Verify the main extracted values.
assert contract_facts["provider"] == "BrightLine Technologies Ltd."
assert contract_facts["client"] == "NovaWare Systems Inc."
assert contract_facts["monthly_fee_cents"] == 1_200_000
assert contract_facts["invoice_due_days"] == 30
assert contract_facts["late_penalty_percent_per_month"] == 2.0
assert contract_facts["term_months"] == 12
assert contract_facts["termination_notice_days"] == 30
assert contract_facts["liability_lookback_months"] == 3
assert contract_facts["governing_law"] == "State of California"

# Verify that every named clause was extracted.
for required_heading in CLAUSE_HEADINGS:
    assert required_heading in contract_sections

# Verify the illustrative liability calculation.
assert illustrative_liability_cap(contract_facts) == 3_600_000

# Verify that a missing topic is not fabricated.
missing_topic_result = assistant.ask(
    "What is the cybersecurity breach notification deadline?"
)
assert "not clearly stated" in missing_topic_result["answer"].lower()

print("All deterministic tests passed.")

# 7. Hallucination, bias and safety controls

## Hallucination controls

- original document as canonical source ;
- explicit clause extraction ;
- “not stated” fallback ;
- no assumptions about Exhibit A ;
- calculations labelled as illustrative ;
- independent critique ;
- deterministic tests.

## Bias controls

Le contrat nomme deux parties organisationnelles. Le système doit appliquer
les mêmes critères de lecture à chaque partie et ne doit pas :

- favoriser automatiquement le Provider ou le Client ;
- supposer qu’une partie est plus puissante ;
- inventer une intention ;
- transformer une obligation contractuelle en jugement moral.

## Legal safety

L’assistant explique le document mais ne remplace pas :

- un avocat ;
- l’accord complet ;
- les faits réels du litige ;
- l’analyse du droit applicable.

# 8. Limites de l’assistant

1. L’entrée est seulement un extrait.
2. Exhibit A n’est pas disponible.
3. Les exceptions, définitions et clauses générales du contrat complet
   peuvent modifier l’analyse.
4. Le parseur est conçu pour les titres de cet exercice.
5. Les réponses déterministes reposent sur des mots-clés.
6. Une critique automatique ne garantit pas une interprétation juridique
   correcte.
7. La mémoire doit être protégée, corrigible et supprimable en production.

# 9. Pistes d’amélioration

Pour une application réelle :

- charger des PDF et documents Word ;
- découper et indexer les clauses dans un vector store ;
- ajouter des citations avec page et paragraphe ;
- détecter les définitions et renvois internes ;
- gérer plusieurs contrats ;
- comparer des versions ;
- ajouter des droits d’accès ;
- chiffrer la mémoire ;
- faire valider les réponses sensibles par un juriste ;
- mesurer la fidélité, la couverture et le taux d’abstention.

# Conclusion

Le notebook met en œuvre un assistant documentaire complet :

```text
Document
   → extraction
   → résumé
   → Q&A adaptative
   → mémoire structurée
   → critique
   → réponse révisable
```

Les points essentiels sont :

- le document doit rester la source prioritaire ;
- la mémoire maintient la continuité sans remplacer la preuve ;
- les calculs doivent expliciter leurs hypothèses ;
- l’assistant doit reconnaître les informations absentes ;
- la critique doit être indépendante de la génération initiale ;
- les domaines juridiques nécessitent une validation professionnelle.